# Deteksi Anomali Transaksi & Kualitas Data SAMSAT (PKB Banten 2021–2024)
**Penelitian 4 – INOVATIF** · Ahmad Dedi Jubaedi, Saleh Dwiyatno (Universitas Serang Raya), Rahmat (AMIK Serang)

Notebook untuk **cek silang di Google Colab** (seed 42; sel-sel di bawah identik dengan `src/pkb_anomaly_pipeline.py`).

**Struktur folder di Google Drive**
```
/content/drive/MyDrive/Penelitian/INOVATIF_Deteksi_Anomali/
├── data/       pkb_2021_2024_input_pseudonim.csv.gz + pkb_inventaris_mentah.csv
│               (atau 4 file 'DATA PKB BULAN MEI - JUNI TAHUN 20xx.xlsx' → set FROM_XLSX = True)
├── src/        pkb_anomaly_pipeline.py, build_manuscript.py, content_id.py, content_en.py, refs.py
└── project4/   ← semua hasil: results/tables, results/figures, results/figures_en, data_processed, manuscript
```
Langkah: *Runtime → Run all* (±10–15 menit di CPU). Bandingkan `project4/results/tables` dengan hasil lokal.

In [ ]:
# [Colab-1] Mount Drive, atur path, dan cek berkas
import sys, os, shutil
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Penelitian/INOVATIF_Deteksi_Anomali'
DATA_DIR = os.path.join(BASE_DIR, 'data')
SRC_DIR  = os.path.join(BASE_DIR, 'src')
OUT_DIR  = os.path.join(BASE_DIR, 'project4')
FROM_XLSX = False      # True = baca 4 file xlsx mentah di DATA_DIR
FAST = False           # True = uji cepat (3 ulangan, sampel kecil)

os.makedirs(OUT_DIR, exist_ok=True)
for d in (DATA_DIR, SRC_DIR):
    assert os.path.isdir(d), f'Folder tidak ditemukan: {d}'
print('data :', sorted(os.listdir(DATA_DIR)))
print('src  :', sorted(os.listdir(SRC_DIR)))
need = ['pkb_2021_2024_input_pseudonim.csv.gz', 'pkb_inventaris_mentah.csv'] if not FROM_XLSX else \
       [f'DATA PKB BULAN MEI - JUNI TAHUN {y}.xlsx' for y in (2021, 2022, 2023, 2024)]
missing = [f for f in need if not os.path.exists(os.path.join(DATA_DIR, f))]
assert not missing, f'Berkas data belum ada di {DATA_DIR}: {missing}'
sys.path.insert(0, SRC_DIR)
!pip -q install openpyxl python-docx
sys.argv = ['pkb_anomaly_pipeline.py', '--data_dir', DATA_DIR, '--out_dir', OUT_DIR] + \
           (['--from_xlsx'] if FROM_XLSX else []) + (['--fast'] if FAST else [])
print('argv:', sys.argv)

> **Alternatif satu langkah:** jalankan langsung skrip di folder `src` (melewati sel-sel bertahap di bawah):
> ```
> !python "$SRC_DIR/pkb_anomaly_pipeline.py" --data_dir "$DATA_DIR" --out_dir "$OUT_DIR"
> ```
> Sel-sel berikut menjalankan langkah yang sama secara bertahap agar setiap tahap dapat diperiksa.

In [ ]:
# -*- coding: utf-8 -*-
"""
Deteksi Anomali Transaksi dan Kualitas Data SAMSAT (PKB Banten)
================================================================
Penelitian 4 - Hybrid rule-based + unsupervised ML (Isolation Forest, LOF, Autoencoder)
Penulis : Ahmad Dedi Jubaedi, Saleh Dwiyatno (Universitas Serang Raya), Rahmat (AMIK Serang)

Data (riil): DATA PKB BULAN MEI - JUNI TAHUN 2021..2024.xlsx (ekspor ISQL-WEB SAMSAT Banten).
Nomor polisi hanya dipakai untuk cek format & duplikasi, lalu diganti pseudonim SHA-256.

Jalankan lokal :  python src/pkb_anomaly_pipeline.py --data_dir "/path/PKB Banten" --out_dir "/path/project4"
Google Colab   :  lihat notebooks/PKB_Anomali_SAMSAT_Colab.ipynb (sel-sel '# %%' di bawah ini)
"""

## [0] Setup

In [ ]:
# %% [0] Setup ---------------------------------------------------------------------------
import os, re, json, time, hashlib, argparse, warnings, platform
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy import stats
import sklearn
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, f1_score

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

ap = argparse.ArgumentParser()
ap.add_argument("--data_dir", default="/mnt/user-data/uploads/PKB Banten")
ap.add_argument("--out_dir", default=os.path.join(os.path.dirname(os.path.abspath(
    globals().get("__file__", os.path.join(os.getcwd(), "src", "x.py")))), ".."))
ap.add_argument("--n_seeds", type=int, default=10)
ap.add_argument("--fast", action="store_true", help="mode cepat (seed & sampel lebih kecil)")
ap.add_argument("--input_csv", default=None, help="dataset terpseudonimisasi (default: <data_dir>/pkb_2021_2024_input_pseudonim.csv.gz)")
ap.add_argument("--from_xlsx", action="store_true", help="paksa baca 4 file xlsx mentah")
ap.add_argument("--export_input", action="store_true", help="simpan dataset terpseudonimisasi untuk Colab")
args, _ = ap.parse_known_args()
DATA, OUT = args.data_dir, os.path.abspath(args.out_dir)
TAB, DP = os.path.join(OUT, "results", "tables"), os.path.join(OUT, "data_processed")
FIGD = {"id": os.path.join(OUT, "results", "figures"), "en": os.path.join(OUT, "results", "figures_en")}
for p in [TAB, DP] + list(FIGD.values()):
    os.makedirs(p, exist_ok=True)
N_SEEDS = 3 if args.fast else args.n_seeds
LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.append(s)
def save(df, name, idx=False):
    df.to_csv(os.path.join(TAB, name + ".csv"), index=idx); return df

PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "legend.frameon": False, "font.family": "DejaVu Sans",
                     "axes.prop_cycle": plt.cycler(color=PAL), "savefig.bbox": "tight"})
def savefig(fig, name, lang):
    fig.savefig(os.path.join(FIGD[lang], name + ".png")); plt.close(fig)
T = lambda lang, i, e: i if lang == "id" else e
YEARS = [2021, 2022, 2023, 2024]
REF_YEARS = [2021, 2022, 2023]          # tahun referensi (profil kode & pelatihan model)
TEST_YEAR = 2024                       # tahun uji temporal


## [1] Muat data mentah & pseudonimisasi

In [ ]:
# %% [1] Muat data mentah & pseudonimisasi -------------------------------------------------
COLS = ['no', 'no_polisi', 'kd_status', 'kd_jen_mohon', 'kd_jen_mutasi', 'kd_lokasi', 'kd_dati2',
        'kd_jenis_kb', 'nm_merek_kb', 'tg_pros_bayar', 'bbnpok', 'bbnden', 'pkbpok', 'pkbden'] + \
       [f"{a}{i}" for i in range(5) for a in ("pkbtgk", "pkbden")]
TGK = [f"pkbtgk{i}" for i in range(5)]; DEN = [f"pkbden{i}" for i in range(5)]
AMT = ["bbnpok", "bbnden", "pkbpok", "pkbden"] + TGK + DEN
def canon_plate(p):
    p = re.sub(r"[^A-Z0-9 ]", " ", str(p).upper()); p = re.sub(r"\s+", " ", p).strip()
    m = re.match(r"^([A-Z]{1,2}) ?(\d{1,4}) ?([A-Z]{0,3})$", p)
    if m: return f"{m[1]}|{int(m[2])}|{m[3]}"
    m = re.match(r"^([A-Z]{1,2}) ([A-Z]{0,3}) ?(\d{1,4})$", p)
    if m: return f"{m[1]}|{int(m[3])}|{m[2]}"
    return None
def pseud(k):  # kunci sama dengan penelitian 3 -> dapat ditautkan
    return hashlib.sha256(("pkb-banten-p3|" + k).encode()).hexdigest()[:16] if isinstance(k, str) else None

t0 = time.time()
CODE_COLS = ["kd_status", "kd_jen_mohon", "kd_jen_mutasi", "kd_lokasi", "kd_dati2", "kd_jenis_kb", "nm_merek_kb"]
IN_CSV = args.input_csv or os.path.join(DATA, "pkb_2021_2024_input_pseudonim.csv.gz")
INV_CSV = os.path.join(os.path.dirname(IN_CSV), "pkb_inventaris_mentah.csv")
if os.path.exists(IN_CSV) and not args.from_xlsx:
    # Mode Colab: dataset terpseudonimisasi (nopol sudah diganti pseudonim SHA-256 & flag format)
    D = pd.read_csv(IN_CSV, dtype={c: str for c in CODE_COLS + ["tg_pros_bayar", "vid"]}, keep_default_na=False, na_values={"vid": [""]})
    inv = pd.read_csv(INV_CSV).to_dict("records")
    log(f"[1] sumber: {os.path.basename(IN_CSV)} (terpseudonimisasi)")
else:
    inv, fr = [], []
    for y in YEARS:
        f = os.path.join(DATA, f"DATA PKB BULAN MEI - JUNI TAHUN {y}.xlsx")
        d = pd.read_excel(f, dtype=str, header=None).iloc[:, :24]; n0 = len(d); d.columns = COLS
        rec = d["no"].astype(str).str.match(r"^\d+\.?$")
        d = d[rec].copy(); d["tahun"] = y
        inv.append({"tahun": y, "baris_mentah": n0, "baris_header_footer": int((~rec).sum()), "transaksi": len(d)})
        fr.append(d)
    D = pd.concat(fr, ignore_index=True)
    for c in AMT:
        D[c] = pd.to_numeric(D[c].astype(str).str.replace(",", ""), errors="coerce").fillna(0)
    for c in CODE_COLS:
        D[c] = D[c].fillna("NA").astype(str).str.strip()
    plate = D["no_polisi"].astype(str).str.upper()
    D["v_plate_fmt"] = (~plate.str.match(r"^[A-Z]{1,2} [A-Z]{1,3}\d{1,4}$")).astype(int)    # format baku ISQL 'B ABC1234'
    D["vid"] = plate.map(canon_plate).map(pseud)
    D = D.drop(columns=["no", "no_polisi"])
    if args.export_input:   # simpan dataset terpseudonimisasi untuk Google Colab
        D.to_csv(IN_CSV, index=False, compression="gzip"); pd.DataFrame(inv).to_csv(INV_CSV, index=False)
        log(f"[1] dataset Colab disimpan: {IN_CSV}")
for c in AMT:
    D[c] = pd.to_numeric(D[c], errors="coerce").fillna(0)
D["tahun"] = D["tahun"].astype(int)
D["tgl"] = pd.to_datetime(D["tg_pros_bayar"], format="%d-%m-%Y", errors="coerce")
D["v_plate_parse"] = D["vid"].isna().astype(int)
D["dup_exact"] = D.duplicated(["vid", "tahun", "tg_pros_bayar", "kd_jen_mohon", "pkbpok", "bbnpok", "pkbden"] + TGK,
                              keep="first").astype(int)
D["n_trx_plate"] = D.groupby(["vid", "tahun"])["vid"].transform("size").fillna(1)
D = D.drop(columns=["tg_pros_bayar"])
D["rid"] = np.arange(len(D))
save(pd.DataFrame(inv), "T01_inventaris_data")
log(f"[1] transaksi: {len(D):,} ({time.time()-t0:.1f}s)")


## [2] Variabel turunan

In [ ]:
# %% [2] Variabel turunan ------------------------------------------------------------------
D["tgk_total"] = D[TGK].sum(axis=1); D["dentgk_total"] = D[DEN].sum(axis=1)
D["thn_tunggak"] = (D[TGK] > 0).sum(axis=1)
D["kelas"] = D["kd_jenis_kb"].str[:1].map({"7": "Sepeda motor", "1": "Mobil penumpang", "2": "Bus/microbus",
                                          "3": "Kendaraan barang", "4": "Kendaraan barang", "5": "Khusus/lainnya"}).fillna("Khusus/lainnya")
# 2b. deteksi pergeseran skema kode (code drift) & harmonisasi lintas tahun
def jsd(p, q):
    idx = p.index.union(q.index); p = p.reindex(idx).fillna(0) + 1e-12; q = q.reindex(idx).fillna(0) + 1e-12
    p, q = p / p.sum(), q / q.sum(); m = (p + q) / 2
    return float(.5 * (p * np.log2(p / m)).sum() + .5 * (q * np.log2(q / m)).sum())
D["pair_raw"] = D["kd_jen_mohon"] + "|" + D["kd_jen_mutasi"]
D["kd_status_raw"], D["kd_jen_mohon_raw"] = D["kd_status"], D["kd_jen_mohon"]
STATUS_MAP = {"5": "3", "6": "4"}                       # kd_status 3/4 (2021-2023) -> 5/6 (2024)
MOHON_MAP = {("2", m): "1" for m in ["7", "9", "5", "6"]}  # daftar ulang+mutasi: 1|x (2021-2023) -> 2|x (2024)
D["kd_status"] = D["kd_status"].replace(STATUS_MAP)
D["kd_jen_mohon"] = [MOHON_MAP.get((a, b), a) for a, b in zip(D["kd_jen_mohon"], D["kd_jen_mutasi"])]
drift = []
for c_raw, c_h, nm in [("kd_status_raw", "kd_status", "kd_status"), ("pair_raw", None, "kd_jen_mohon|kd_jen_mutasi"),
                       ("kd_jenis_kb", "kd_jenis_kb", "kd_jenis_kb"), ("kd_lokasi", "kd_lokasi", "kd_lokasi"), ("kd_dati2", "kd_dati2", "kd_dati2")]:
    for y in YEARS:
        refm = D.tahun.isin([x for x in REF_YEARS if x != y])
        if c_h is None:
            h = D["kd_jen_mohon"] + "|" + D["kd_jen_mutasi"]
        else:
            h = D[c_h]
        pr, qr = D.loc[D.tahun == y, c_raw].value_counts(normalize=True), D.loc[refm, c_raw].value_counts(normalize=True)
        ph, qh = h[D.tahun == y].value_counts(normalize=True), h[refm].value_counts(normalize=True)
        drift.append({"field": nm, "tahun": y, "JSD_mentah": jsd(pr, qr), "JSD_harmonisasi": jsd(ph, qh),
                      "kode_baru": ",".join(sorted(set(pr.index) - set(qr.index))) or "-",
                      "kode_hilang": ",".join(sorted(set(qr.index) - set(pr.index))) or "-"})
DRIFT = pd.DataFrame(drift)
save(DRIFT.round(4), "T03b_pergeseran_kode")
D["pair"] = D["kd_jen_mohon"] + "|" + D["kd_jen_mutasi"]
D["lok_dati"] = D["kd_lokasi"] + "|" + D["kd_dati2"]
D["hari_ke"] = (D["tgl"] - pd.to_datetime(D["tahun"].astype(str) + "-05-01")).dt.days
D["dow"] = D["tgl"].dt.dayofweek
REF = D["tahun"].isin(REF_YEARS)


## [3] Mesin aturan domain (rule engine)

In [ ]:
# %% [3] Mesin aturan domain (rule engine) ---------------------------------------------------
# 3a. profil kode dari tahun referensi (kamus kode resmi tidak tersedia -> diturunkan dari data)
mohon_n = D[REF].groupby("kd_jen_mohon").size()
pair_ref = D[REF].groupby(["kd_jen_mohon", "pair"]).agg(n=("rid", "size"), bbn_rate=("bbnpok", lambda s: (s > 0).mean())).reset_index()
pair_ref["share_in_mohon"] = pair_ref["n"] / pair_ref["kd_jen_mohon"].map(mohon_n)
RARE_PAIR = set(pair_ref.loc[(pair_ref.share_in_mohon < 0.005) | (pair_ref.n < 30), "pair"])
BBN_REQ = set(pair_ref.loc[(pair_ref.bbn_rate >= .95) & (pair_ref.n >= 30), "pair"])
BBN_NONE = set(pair_ref.loc[(pair_ref.bbn_rate <= .05) & (pair_ref.n >= 30), "pair"])
code_cnt = {c: D[REF][c].value_counts() for c in ["kd_jen_mohon", "kd_jen_mutasi", "kd_jenis_kb"]}
save(pair_ref.sort_values(["kd_jen_mohon", "n"], ascending=[True, False]).assign(
    status=lambda x: np.where(x.pair.isin(RARE_PAIR), "langka/tidak konsisten", "lazim")), "S01_profil_kode_mohon_mutasi")

# 3b. profil nilai PKB per jenis kendaraan (median & MAD log10 pokok)
lp = np.log10(D["pkbpok"].where(D["pkbpok"] > 0))
grp_stats = pd.DataFrame({"med": lp[REF].groupby(D.loc[REF, "kd_jenis_kb"]).median(),
                          "mad": lp[REF].groupby(D.loc[REF, "kd_jenis_kb"]).apply(lambda s: stats.median_abs_deviation(s.dropna(), scale="normal")),
                          "n": lp[REF].groupby(D.loc[REF, "kd_jenis_kb"]).count()})
cls_stats = pd.DataFrame({"med": lp[REF].groupby(D.loc[REF, "kelas"]).median(),
                          "mad": lp[REF].groupby(D.loc[REF, "kelas"]).apply(lambda s: stats.median_abs_deviation(s.dropna(), scale="normal"))})
def robust_z(df):
    l = np.log10(df["pkbpok"].where(df["pkbpok"] > 0))
    g = grp_stats.reindex(df["kd_jenis_kb"]); c = cls_stats.reindex(df["kelas"])
    use_g = (g["n"].values >= 30) & (g["mad"].values > 0)
    med = np.where(use_g, g["med"].values, c["med"].values); mad = np.where(use_g, g["mad"].values, c["mad"].values)
    return pd.Series((l.values - med) / np.maximum(mad, 0.05), index=df.index).fillna(0)

RULES = [  # id, nama (ID), nama (EN), dimensi DQ, bobot
    ("R01", "Denda tanpa tunggakan", "Penalty without arrears", "Konsistensi", 3),
    ("R02", "Tunggakan tanpa denda", "Arrears without penalty", "Konsistensi", 2),
    ("R03", "Tarif denda tunggakan menyimpang", "Off-schedule arrears penalty rate", "Akurasi", 2),
    ("R04", "Denda berjalan >25% / tanpa pokok", "Current penalty >25% or without principal", "Akurasi", 2),
    ("R05", "Pola tunggakan tidak berurutan", "Non-contiguous arrears pattern", "Konsistensi", 2),
    ("R06", "Tunggakan tanpa pokok berjalan", "Arrears with zero current principal", "Konsistensi", 2),
    ("R07", "Kode permohonan-mutasi tidak konsisten", "Inconsistent application-mutation code", "Konsistensi", 3),
    ("R08", "BBNKB tidak konsisten dengan kode", "Transfer fee (BBNKB) inconsistent with code", "Konsistensi", 3),
    ("R09", "Nilai PKB janggal untuk jenis kendaraan", "Implausible PKB value for vehicle type", "Plausibilitas", 2),
    ("R10", "Rasio tunggakan/pokok ekstrem", "Extreme arrears-to-principal ratio", "Plausibilitas", 1),
    ("R11", "Duplikat transaksi persis", "Exact duplicate transaction", "Keunikan", 2),
]
RULE_IDS = [r[0] for r in RULES]; RW = {r[0]: r[4] for r in RULES}
RNAME = {"id": {r[0]: r[1] for r in RULES}, "en": {r[0]: r[2] for r in RULES}}

def apply_rules(df):
    tg, dn = df[TGK].values, df[DEN].values
    with np.errstate(divide="ignore", invalid="ignore"):
        rr = np.where(tg > 0, dn / tg, np.nan)
        rden = np.where(df["pkbpok"] > 0, df["pkbden"] / df["pkbpok"], np.nan)
        rtgk = np.where((df["pkbpok"] > 0) & (df["pkbtgk0"] > 0), df["pkbtgk0"] / df["pkbpok"], np.nan)
    has = tg > 0
    out = pd.DataFrame(index=df.index)
    out["R01"] = ((dn > 0) & (tg == 0)).any(axis=1)
    out["R02"] = ((tg > 0) & (dn == 0)).any(axis=1)
    dev0 = rr[:, 0] > 0.2525
    devk = ((rr[:, 1:] > 0) & ((rr[:, 1:] < 0.245) | (rr[:, 1:] > 0.2525))).any(axis=1)
    out["R03"] = dev0 | devk
    out["R04"] = (rden > 0.2525) | ((df["pkbden"] > 0) & (df["pkbpok"] <= 0))
    out["R05"] = np.array([any((not r[j]) and r[j + 1:].any() for j in range(4)) for r in has])
    out["R06"] = (df["tgk_total"] > 0) & (df["pkbpok"] <= 0)
    rare_code = df["kd_jen_mohon"].map(code_cnt["kd_jen_mohon"]).fillna(0).lt(50) | \
                df["kd_jen_mutasi"].map(code_cnt["kd_jen_mutasi"]).fillna(0).lt(50)
    out["R07"] = df["pair"].isin(RARE_PAIR) | ~df["pair"].isin(set(pair_ref["pair"])) | rare_code
    out["R08"] = (df["pair"].isin(BBN_REQ) & (df["bbnpok"] <= 0)) | (df["pair"].isin(BBN_NONE) & (df["bbnpok"] > 0)) | \
                 ((df["bbnden"] > 0) & (df["bbnpok"] <= 0))
    out["R09"] = robust_z(df).abs() > 3.5
    out["R10"] = (rtgk > 3) | (rtgk < 0.3)
    out["R11"] = df["dup_exact"].astype(bool) if "dup_exact" in df else False
    out = out.astype(int)
    out["rule_score"] = sum(out[r] * RW[r] for r in RULE_IDS)
    out["rule_flag"] = (out[RULE_IDS].sum(axis=1) > 0).astype(int)
    return out

RU = apply_rules(D)
D = pd.concat([D, RU], axis=1)
D["zero_value"] = (D[AMT].sum(axis=1) == 0).astype(int)        # informatif (bukan pelanggaran)
log(f"[3] rule flag: {D.rule_flag.sum():,} ({D.rule_flag.mean():.2%}); nol-nilai {D.zero_value.sum():,}")

# T02 katalog aturan
logic = {"R01": "pkbden_k>0 dan pkbtgk_k=0 (k=0..4)", "R02": "pkbtgk_k>0 dan pkbden_k=0",
         "R03": "denda_k/tunggakan_k di luar [0,245;0,2525] (k>=1) atau >0,2525 (k=0)",
         "R04": "pkbden/pkbpok>0,2525 atau pkbden>0 dengan pkbpok=0", "R05": "tunggakan tahun k+1 ada, tahun k kosong",
         "R06": "total tunggakan>0 dan pkbpok=0",
         "R07": "pasangan (kd_jen_mohon,kd_jen_mutasi) <0,5% dalam kode permohonan / n<30, atau kode langka (<50)",
         "R08": "pasangan wajib BBN (>=95%) tanpa BBN, pasangan tanpa BBN (<=5%) ber-BBN, atau bbnden>0 tanpa bbnpok",
         "R09": "|robust-z log10 pkbpok dalam kd_jenis_kb|>3,5 (median/MAD tahun referensi)",
         "R10": "pkbtgk0/pkbpok >3 atau <0,3", "R11": "duplikat persis (nopol, tanggal, kode, nilai)"}
save(pd.DataFrame([{"ID": r[0], "Aturan": r[1], "Rule": r[2], "Dimensi": r[3], "Bobot": r[4], "Logika": logic[r[0]]} for r in RULES]),
     "T02_katalog_aturan")


## [4] Profil kualitas data per tahun

In [ ]:
# %% [4] Profil kualitas data per tahun -------------------------------------------------------
rows, rv = [], []
for y in YEARS:
    s = D[D.tahun == y]; n = len(s)
    r = {"tahun": y, "n": n}
    for k in RULE_IDS:
        r[k] = int(s[k].sum()); r[k + "_pct"] = 100 * s[k].mean()
        rv.append({"tahun": y, "aturan": k, "n": int(s[k].sum()),
                   "nilai_transaksi_rp": float((s.loc[s[k] == 1, ["pkbpok", "pkbden", "bbnpok", "bbnden", "tgk_total", "dentgk_total"]].sum(axis=1)).sum())})
    r["rule_flag"] = int(s.rule_flag.sum()); r["rule_flag_pct"] = 100 * s.rule_flag.mean()
    r["zero_value"] = int(s.zero_value.sum()); r["zero_value_pct"] = 100 * s.zero_value.mean()
    rows.append(r)
T03 = save(pd.DataFrame(rows), "T03_prevalensi_aturan_per_tahun")
save(pd.DataFrame(rv), "S02_nilai_transaksi_terflag_per_aturan")

def dq_score(s):
    comp = 100 * (1 - s[["kd_jenis_kb", "nm_merek_kb", "kd_lokasi", "kd_dati2", "kd_jen_mohon"]].eq("NA").mean().mean())
    valid = 100 * (1 - ((s.v_plate_fmt == 1) | s.tgl.isna() | ~s.tgl.dt.month.isin([5, 6])).mean())
    cons = 100 * (1 - s[["R01", "R02", "R05", "R06", "R07", "R08"]].max(axis=1).mean())
    acc = 100 * (1 - s[["R03", "R04"]].max(axis=1).mean())
    plaus = 100 * (1 - s[["R09", "R10"]].max(axis=1).mean())
    uniq = 100 * (1 - s.R11.mean())
    return {"Kelengkapan": comp, "Validitas": valid, "Konsistensi": cons, "Akurasi": acc, "Plausibilitas": plaus, "Keunikan": uniq}
T04 = pd.DataFrame({y: dq_score(D[D.tahun == y]) for y in YEARS}).T
T04["Indeks DQ"] = T04.mean(axis=1)
save(T04.round(3).reset_index().rename(columns={"index": "tahun"}), "T04_skor_kualitas_data")
log("[4] DQ index:", T04["Indeks DQ"].round(2).to_dict())


## [5] Rekayasa fitur ML

In [ ]:
# %% [5] Rekayasa fitur ML ------------------------------------------------------------------
STAT = sorted(D["kd_status"].unique())
_lr = np.log10((D["bbnpok"] / D["pkbpok"].where(D["pkbpok"] > 0)).where(D["bbnpok"] > 0))
_m = REF & _lr.notna() & (D.rule_flag == 0)
bbn_stats = pd.DataFrame({"med": _lr[_m].groupby(D.loc[_m, "pair"]).median(),
                          "mad": _lr[_m].groupby(D.loc[_m, "pair"]).apply(lambda v: stats.median_abs_deviation(v, scale="normal"))})
freq_maps = {c: D[REF][c].value_counts(normalize=True) for c in ["kd_jenis_kb", "kd_jen_mohon", "kd_jen_mutasi", "pair"]}
def make_features(df, domain=True):
    X = pd.DataFrame(index=df.index)
    for c in ["pkbpok", "pkbden", "bbnpok", "bbnden", "tgk_total", "dentgk_total"]:
        X["log_" + c] = np.log1p(df[c].clip(lower=0))
    X["thn_tunggak"] = df["thn_tunggak"]
    for c in ["kd_jenis_kb", "kd_jen_mohon", "kd_jen_mutasi"]:
        X["freq_" + c] = np.log10(df[c].map(freq_maps[c]).fillna(1e-6))
    for s in STAT:
        X["st_" + s] = (df["kd_status"] == s).astype(int)
    if domain:   # fitur berbasis pengetahuan domain
        pk = df["pkbpok"].where(df["pkbpok"] > 0)
        X["r_den"] = (df["pkbden"] / pk).fillna(0).clip(0, 2)
        X["r_dentgk"] = (df["dentgk_total"] / df["tgk_total"].where(df["tgk_total"] > 0)).fillna(0).clip(0, 2)
        X["r_tgk"] = np.log1p((df["tgk_total"] / pk).fillna(0).clip(0, 50))
        X["r_bbn"] = np.log1p((df["bbnpok"] / pk).fillna(0).clip(0, 500))
        X["z_jenis"] = robust_z(df).clip(-15, 15)
        X["freq_pair"] = np.log10(df["pair"].map(freq_maps["pair"]).fillna(1e-6))
        X["bbn_x_pair"] = (df["bbnpok"] > 0).astype(int) - df["pair"].map(pair_ref.set_index("pair")["bbn_rate"]).fillna(0.5)
        X["den_wo_tgk"] = ((df[DEN].values > 0) & (df[TGK].values == 0)).sum(axis=1)
        lr = np.log10((df["bbnpok"] / df["pkbpok"].where(df["pkbpok"] > 0)).where(df["bbnpok"] > 0))
        st = bbn_stats.reindex(df["pair"])
        X["z_bbn"] = ((lr.values - st["med"].values) / np.maximum(st["mad"].values, .05))
        X["z_bbn"] = X["z_bbn"].fillna(0).clip(-15, 15)
    return X.astype(float)
FEAT_DOMAIN = list(make_features(D.head(50)).columns); FEAT_RAW = list(make_features(D.head(50), False).columns)
save(pd.DataFrame({"fitur": FEAT_DOMAIN, "kelompok": ["domain" if f not in FEAT_RAW else "dasar" for f in FEAT_DOMAIN]}), "T05_daftar_fitur")


## [6] Detektor

In [ ]:
# %% [6] Detektor ---------------------------------------------------------------------------
class Detectors:
    """IF, LOF (novelty) dan Autoencoder (MLP) dilatih pada transaksi bersih tahun referensi."""
    def __init__(self, seed=SEED, domain=True, if_n=300, lof_k=35, ae_hidden=(32, 8, 32), n_fit=40000):
        self.seed, self.domain, self.if_n, self.lof_k, self.ae_hidden, self.n_fit = seed, domain, if_n, lof_k, ae_hidden, n_fit
    def fit(self, df):
        rng = np.random.RandomState(self.seed)
        idx = rng.choice(len(df), min(self.n_fit, len(df)), replace=False)
        X = make_features(df.iloc[idx], self.domain)
        self.cols = X.columns; self.sc = StandardScaler().fit(X)
        Z = np.clip(self.sc.transform(X), -10, 10)
        tm = {}
        t = time.time(); self.iforest = IsolationForest(n_estimators=self.if_n, max_samples=512, random_state=self.seed, n_jobs=-1).fit(Z); tm["IF"] = time.time() - t
        t = time.time(); self.lof = LocalOutlierFactor(n_neighbors=self.lof_k, novelty=True, n_jobs=-1).fit(Z[:20000]); tm["LOF"] = time.time() - t
        t = time.time(); self.ae = MLPRegressor(hidden_layer_sizes=self.ae_hidden, activation="relu", alpha=1e-4, batch_size=256,
                                                learning_rate_init=1e-3, max_iter=80, early_stopping=True, n_iter_no_change=6,
                                                random_state=self.seed).fit(Z, Z); tm["AE"] = time.time() - t
        self.fit_time = tm
        self.ref = {k: np.sort(v) for k, v in self._raw(Z).items()}
        rz = self._raw(Z)
        self.ref["ENS"] = np.sort(self._fuse({k: self._pct(k, v) for k, v in rz.items()}))
        self.ref["ENS_MEAN"] = np.sort(np.mean([self._pct(k, v) for k, v in rz.items()], axis=0))
        return self
    def _raw(self, Z):
        return {"IF": -self.iforest.score_samples(Z), "LOF": -self.lof.score_samples(Z),
                "AE": ((self.ae.predict(Z) - Z) ** 2).mean(axis=1)}
    @staticmethod
    def _fuse(P):
        """Fusi maksimum persentil (tie-break: rerata persentil) - lihat Aggarwal & Sathe (outlier ensembles)."""
        A = np.vstack([P[k] for k in ["IF", "LOF", "AE"]])
        return A.max(axis=0) + 1e-3 * A.mean(axis=0)
    def _pct(self, k, v):
        return np.searchsorted(self.ref[k], v, side="right") / len(self.ref[k])
    def score(self, df):
        Z = np.clip(self.sc.transform(make_features(df, self.domain)[self.cols]), -10, 10)
        out, tm = {}, {}
        for k in ["IF", "LOF", "AE"]:
            t = time.time()
            v = {"IF": lambda: -self.iforest.score_samples(Z), "LOF": lambda: -self.lof.score_samples(Z),
                 "AE": lambda: ((self.ae.predict(Z) - Z) ** 2).mean(axis=1)}[k]()
            tm[k] = time.time() - t; out[k] = v
        P = {k: self._pct(k, out[k]) for k in ["IF", "LOF", "AE"]}
        out["ENS"] = self._fuse(P); out["ENS_MEAN"] = np.mean([P[k] for k in P], axis=0)
        self.score_time = tm; self._Z = Z
        return pd.DataFrame(out, index=df.index)
    def flags(self, S, q=0.99):
        return pd.DataFrame({k: (S[k] > np.quantile(self.ref[k], q)).astype(int) for k in ["IF", "LOF", "AE", "ENS"]}, index=S.index)
    def reasons(self, idx_rows, top=3):
        E = (self.ae.predict(self._Z[idx_rows]) - self._Z[idx_rows]) ** 2
        return [", ".join(self.cols[np.argsort(-e)[:top]]) for e in E]

def hybrid_score(S, rules):
    """Kaskade: transaksi yang melanggar aturan diprioritaskan; urutan di dalamnya & sisanya oleh skor ensemble ML."""
    return rules["rule_flag"].values * 1.0 + S["ENS"].values


## [7] Injeksi anomali terkendali (ground truth sintetis)

In [ ]:
# %% [7] Injeksi anomali terkendali (ground truth sintetis) --------------------------------------
INJ = [  # kode, nama ID, nama EN, aturan yang relevan
    ("A1", "Denda tanpa tunggakan", "Penalty without arrears", "R01"),
    ("A2", "Kode mutasi tidak konsisten", "Inconsistent mutation code", "R07"),
    ("A3", "Nilai PKB lintas jenis", "Cross-type PKB value", "R09 (sebagian)"),
    ("A4", "Salah digit nilai pokok (x10 / /10)", "Digit error in principal (x10 or /10)", "R09/R04 (sebagian)"),
    ("A5", "Kode jenis kendaraan tertukar", "Swapped vehicle-type code", "R09"),
    ("A6", "Tarif denda tunggakan salah", "Wrong arrears penalty rate", "R03"),
    ("A7", "Tunggakan menggelembung (x2,5-5)", "Inflated arrears (x2.5-5)", "R10 (sebagian)"),
    ("A8", "Nilai BBNKB janggal (x/÷ 5-12)", "Implausible transfer fee (x or / 5-12)", "- (tanpa aturan)"),
]
def inject(base, pool_tgk, pool_bbn, n_per, rng):
    """base: transaksi bersih tahun uji. Menghasilkan salinan termodifikasi + label tipe."""
    car_pok = base.loc[base.kelas == "Mobil penumpang", "pkbpok"].values
    moto_pok = base.loc[base.kelas == "Sepeda motor", "pkbpok"].values
    out = []
    def take(src, n):
        return src.iloc[rng.choice(len(src), n, replace=False)].copy()
    a = take(base[base.pkbpok > 0], n_per)                                           # A1
    a["pkbden1"] = np.round(a["pkbpok"] * rng.uniform(.05, .25, n_per), -2); a["typ"] = "A1"; out.append(a)
    a = take(base, n_per)                                                             # A2
    a["kd_jen_mutasi"] = np.where(a["kd_jen_mohon"].isin(["1", "2"]), rng.choice(["1", "2", "6"], n_per),
                         np.where(a["kd_jen_mohon"] == "3", "7", "NA")); a["typ"] = "A2"; out.append(a)
    a = take(base[base.kelas.isin(["Sepeda motor", "Mobil penumpang"]) & (base.pkbpok > 0)], n_per)   # A3
    r = a["pkbden"] / a["pkbpok"]
    a["pkbpok"] = np.where(a.kelas == "Sepeda motor", rng.choice(car_pok, n_per), rng.choice(moto_pok, n_per))
    a["pkbden"] = np.round(a["pkbpok"] * r, -2); a["typ"] = "A3"; out.append(a)
    a = take(base[base.pkbpok > 0], n_per)                                           # A4
    a["pkbpok"] = np.round(np.where(rng.rand(n_per) < .5, a["pkbpok"] * 10, a["pkbpok"] / 10), -2); a["typ"] = "A4"; out.append(a)
    a = take(base[base.kd_jenis_kb.isin(["701", "103", "102", "301"])], n_per)        # A5
    a["kd_jenis_kb"] = np.where(a.kd_jenis_kb == "701", rng.choice(["103", "102", "301"], n_per), "701"); a["typ"] = "A5"; out.append(a)
    a = take(pool_tgk, n_per)                                                         # A6
    rate = np.where(rng.rand(n_per) < .5, rng.uniform(.05, .18, n_per), rng.uniform(.32, .5, n_per))
    for k in range(5):
        a[f"pkbden{k}"] = np.round(a[f"pkbtgk{k}"] * rate, -2)
    a["typ"] = "A6"; out.append(a)
    a = take(pool_tgk, n_per)                                                         # A7
    m = rng.uniform(2.5, 5, n_per)
    for k in range(5):
        a[f"pkbtgk{k}"] = np.round(a[f"pkbtgk{k}"] * m, -2); a[f"pkbden{k}"] = np.round(a[f"pkbden{k}"] * m, -2)
    a["typ"] = "A7"; out.append(a)
    a = take(pool_bbn, n_per)                                                         # A8
    m = rng.uniform(5, 12, n_per); a["bbnpok"] = np.round(np.where(rng.rand(n_per) < .5, a["bbnpok"] * m, a["bbnpok"] / m), -2)
    a["typ"] = "A8"; out.append(a)
    A = pd.concat(out)
    # hitung ulang variabel turunan setelah modifikasi
    A["tgk_total"] = A[TGK].sum(axis=1); A["dentgk_total"] = A[DEN].sum(axis=1); A["thn_tunggak"] = (A[TGK] > 0).sum(axis=1)
    A["kelas"] = A["kd_jenis_kb"].str[:1].map({"7": "Sepeda motor", "1": "Mobil penumpang", "2": "Bus/microbus",
                                              "3": "Kendaraan barang", "4": "Kendaraan barang", "5": "Khusus/lainnya"}).fillna("Khusus/lainnya")
    A["pair"] = A["kd_jen_mohon"] + "|" + A["kd_jen_mutasi"]; A["dup_exact"] = 0
    return A

def evaluate(y, s, flag, k):
    order = np.argsort(-s); topk = y[order[:k]]; top5 = y[order[:int(.05 * len(y))]]
    return {"ROC_AUC": roc_auc_score(y, s), "PR_AUC": average_precision_score(y, s), "P@k": topk.mean(),
            "Recall@5%": top5.sum() / y.sum(), "F1": f1_score(y, flag), "Presisi": (y[flag == 1].mean() if flag.sum() else 0),
            "Recall": flag[y == 1].mean()}

CLEAN = D[(D.rule_flag == 0) & (D.zero_value == 0)]
TRAIN = CLEAN[CLEAN.tahun.isin(REF_YEARS)].reset_index(drop=True)
TESTB = CLEAN[CLEAN.tahun == TEST_YEAR].reset_index(drop=True)
POOL_TGK = TESTB[TESTB.tgk_total > 0]; POOL_BBN = TESTB[TESTB.bbnpok > 0]
N_NORM, N_PER = (8000, 60) if args.fast else (20000, 100)
log(f"[7] train bersih {len(TRAIN):,} | uji bersih {len(TESTB):,} | pool tunggakan {len(POOL_TGK):,} | pool BBN {len(POOL_BBN):,}")
save(pd.DataFrame([{"Kode": i[0], "Tipe anomali": i[1], "Anomaly type": i[2], "Aturan terkait": i[3], "n per ulangan": N_PER} for i in INJ]),
     "T06_tipe_injeksi_anomali")

def build_test(seed):
    rng = np.random.RandomState(1000 + seed)
    norm = TESTB.iloc[rng.choice(len(TESTB), N_NORM, replace=False)].copy(); norm["typ"] = "normal"
    A = inject(TESTB, POOL_TGK, POOL_BBN, N_PER, rng)
    te = pd.concat([norm, A], ignore_index=True); te["y"] = (te.typ != "normal").astype(int)
    te = te.sample(frac=1, random_state=seed).reset_index(drop=True)      # acak urutan (hindari bias ties)
    return te

METHODS = ["Aturan", "IF", "LOF", "AE", "Ensemble ML", "Hibrida"]
MLAB = {"id": dict(zip(METHODS, METHODS)), "en": dict(zip(METHODS, ["Rules", "IF", "LOF", "AE", "ML ensemble", "Hybrid"]))}


## [8] Eksperimen utama: 10 ulangan injeksi + temporal (latih 2021-2023, uji 2024)

In [ ]:
# %% [8] Eksperimen utama: 10 ulangan injeksi + temporal (latih 2021-2023, uji 2024) ---------------
res, typ_rows, curves, times = [], [], {}, []
for sd in range(N_SEEDS):
    te = build_test(sd)
    rl = apply_rules(te)
    det = Detectors(seed=sd).fit(TRAIN)
    S = det.score(te); F = det.flags(S)
    scores = {"Aturan": rl["rule_score"].values.astype(float), "IF": S["IF"].values, "LOF": S["LOF"].values,
              "AE": S["AE"].values, "Ensemble ML": S["ENS"].values, "Hibrida": hybrid_score(S, rl)}
    flags = {"Aturan": rl["rule_flag"].values, "IF": F["IF"].values, "LOF": F["LOF"].values, "AE": F["AE"].values,
             "Ensemble ML": F["ENS"].values, "Hibrida": np.maximum(rl["rule_flag"].values, F["ENS"].values)}
    y = te["y"].values; k = int(y.sum())
    for m in METHODS:
        r = evaluate(y, scores[m], flags[m], k); r.update({"seed": sd, "metode": m}); res.append(r)
        order = np.argsort(-scores[m]); top = np.zeros(len(y), int); top[order[:k]] = 1
        for t in [i[0] for i in INJ]:
            msk = te.typ.values == t
            typ_rows.append({"seed": sd, "metode": m, "tipe": t, "recall_at_k": top[msk].mean(), "recall_flag": flags[m][msk].mean()})
    if sd == 0:
        curves = {m: (y, scores[m]) for m in METHODS}
        te0, S0, rl0, det0 = te, S, rl, det
        fpr_norm = {m: flags[m][y == 0].mean() for m in METHODS}
    times.append({"seed": sd, **{f"fit_{k_}": v for k_, v in det.fit_time.items()}, **{f"score_{k_}": v for k_, v in det.score_time.items()}})
    log(f"   seed {sd}: " + " | ".join(f"{m} AP={average_precision_score(y, scores[m]):.3f}" for m in METHODS))
R8 = pd.DataFrame(res); TY = pd.DataFrame(typ_rows); TM = pd.DataFrame(times)
save(R8, "S03_hasil_per_seed"); save(TY, "S04_recall_per_tipe_per_seed"); save(TM, "S05_waktu_komputasi")
met = ["ROC_AUC", "PR_AUC", "P@k", "Recall@5%", "Presisi", "Recall", "F1"]
agg = R8.groupby("metode")[met].agg(["mean", "std"]).reindex(METHODS)
T07 = pd.DataFrame({m_: [f"{agg.loc[x, (m_, 'mean')]:.3f} ± {agg.loc[x, (m_, 'std')]:.3f}" for x in METHODS] for m_ in met}, index=METHODS)
T07["FPR (normal)"] = [f"{fpr_norm[x]:.3%}" for x in METHODS]
save(T07.reset_index().rename(columns={"index": "Metode"}), "T07_perbandingan_metode")
save(agg.round(4).reset_index(), "S06_perbandingan_metode_numerik")
T08 = TY.groupby(["tipe", "metode"]).recall_at_k.mean().unstack()[METHODS]
save(T08.round(3).reset_index(), "T08_recall_per_tipe_anomali")
log("[8] PR-AUC mean:", agg[("PR_AUC", "mean")].round(3).to_dict())

# uji statistik: Friedman + Wilcoxon (Hibrida vs lainnya, koreksi Holm)
piv = R8.pivot(index="seed", columns="metode", values="PR_AUC")[METHODS]
fr_stat, fr_p = stats.friedmanchisquare(*[piv[m] for m in METHODS])
wil = []
for m in METHODS[:-1]:
    try: w, p = stats.wilcoxon(piv["Hibrida"], piv[m])
    except ValueError: w, p = np.nan, 1.0
    wil.append({"pembanding": m, "selisih_PR_AUC_rerata": (piv["Hibrida"] - piv[m]).mean(), "W": w, "p": p,
                "menang/seri/kalah": f"{(piv['Hibrida'] > piv[m]).sum()}/{(piv['Hibrida'] == piv[m]).sum()}/{(piv['Hibrida'] < piv[m]).sum()}"})
wil = pd.DataFrame(wil).sort_values("p")
wil["p_holm"] = np.minimum(1, np.maximum.accumulate(wil["p"].values * (len(wil) - np.arange(len(wil)))))
wil = wil.sort_index(); wil["Friedman_chi2"] = fr_stat; wil["Friedman_p"] = fr_p
save(wil.round(5), "T09_uji_statistik")
log(f"[8] Friedman chi2={fr_stat:.2f} p={fr_p:.2e}")


## [9] Analisis sensitivitas & ablasi fitur (seed 0)

In [ ]:
# %% [9] Analisis sensitivitas & ablasi fitur (seed 0) ----------------------------------------------
y0 = te0["y"].values; sens = []
for kk in [10, 20, 35, 50, 100]:
    d_ = Detectors(seed=0, lof_k=kk).fit(TRAIN); s_ = d_.score(te0)
    sens.append({"parameter": "LOF n_neighbors", "nilai": kk, "metode": "LOF", "PR_AUC": average_precision_score(y0, s_["LOF"]),
                 "PR_AUC_hibrida": average_precision_score(y0, hybrid_score(s_, rl0))})
for nn in [50, 100, 300, 500]:
    d_ = Detectors(seed=0, if_n=nn).fit(TRAIN); s_ = d_.score(te0)
    sens.append({"parameter": "IF n_estimators", "nilai": nn, "metode": "IF", "PR_AUC": average_precision_score(y0, s_["IF"]),
                 "PR_AUC_hibrida": average_precision_score(y0, hybrid_score(s_, rl0))})
for hb in [(32, 4, 32), (32, 8, 32), (32, 12, 32), (64, 16, 64)]:
    d_ = Detectors(seed=0, ae_hidden=hb).fit(TRAIN); s_ = d_.score(te0)
    sens.append({"parameter": "AE arsitektur", "nilai": "-".join(map(str, hb)), "metode": "AE", "PR_AUC": average_precision_score(y0, s_["AE"]),
                 "PR_AUC_hibrida": average_precision_score(y0, hybrid_score(s_, rl0))})
for nf in [5000, 10000, 20000, 40000]:
    d_ = Detectors(seed=0, n_fit=nf).fit(TRAIN); s_ = d_.score(te0)
    sens.append({"parameter": "Ukuran data latih", "nilai": nf, "metode": "Ensemble ML", "PR_AUC": average_precision_score(y0, s_["ENS"]),
                 "PR_AUC_hibrida": average_precision_score(y0, hybrid_score(s_, rl0))})
d_ = Detectors(seed=0).fit(TRAIN); s_ = d_.score(te0)
for nm_, col_ in [("max persentil (dipakai)", "ENS"), ("rerata persentil", "ENS_MEAN")]:
    s2 = s_.copy(); s2["ENS"] = s_[col_]
    sens.append({"parameter": "Strategi fusi", "nilai": nm_, "metode": "Ensemble ML", "PR_AUC": average_precision_score(y0, s_[col_]),
                 "PR_AUC_hibrida": average_precision_score(y0, hybrid_score(s2, rl0))})
T10 = save(pd.DataFrame(sens).round(4), "T10_sensitivitas")

abl = []
for sd in range(min(N_SEEDS, 5)):
    te = build_test(sd); rl = apply_rules(te); y = te["y"].values
    for dom in [False, True]:
        d_ = Detectors(seed=sd, domain=dom).fit(TRAIN); s_ = d_.score(te)
        for m, col in [("IF", "IF"), ("LOF", "LOF"), ("AE", "AE"), ("Ensemble ML", "ENS")]:
            abl.append({"seed": sd, "fitur": "dasar+domain" if dom else "dasar", "metode": m, "PR_AUC": average_precision_score(y, s_[col])})
        abl.append({"seed": sd, "fitur": "dasar+domain" if dom else "dasar", "metode": "Hibrida", "PR_AUC": average_precision_score(y, hybrid_score(s_, rl))})
ABL = pd.DataFrame(abl)
T11 = ABL.groupby(["metode", "fitur"]).PR_AUC.agg(["mean", "std"]).unstack().round(4)
save(T11.reset_index(), "T11_ablasi_fitur")
log("[9] ablasi selesai")


## [10] Penerapan pada seluruh data riil 2021-2024

In [ ]:
# %% [10] Penerapan pada seluruh data riil 2021-2024 ------------------------------------------
detF = Detectors(seed=SEED).fit(TRAIN)
SA = detF.score(D); FA = detF.flags(SA)
D["s_IF"], D["s_LOF"], D["s_AE"], D["s_ENS"] = SA["IF"].values, SA["LOF"].values, SA["AE"].values, SA["ENS"].values
D["ml_flag"] = FA["ENS"].values
D["hyb_score"] = hybrid_score(SA, D)
D["kategori"] = np.select([(D.rule_flag == 1) & (D.ml_flag == 1), D.rule_flag == 1, D.zero_value == 1, D.ml_flag == 1],
                          ["Aturan & ML", "Aturan saja", "Nihil (informatif)", "ML saja"], "Normal")
T12 = D.groupby("tahun").agg(n=("rid", "size"), aturan=("rule_flag", "sum"), ml=("ml_flag", "sum"),
                             keduanya=("kategori", lambda s: (s == "Aturan & ML").sum()),
                             ml_saja=("kategori", lambda s: (s == "ML saja").sum()),
                             aturan_saja=("kategori", lambda s: (s == "Aturan saja").sum()),
                             nihil=("kategori", lambda s: (s == "Nihil (informatif)").sum())).reset_index()
D["nilai_rp"] = D[["pkbpok", "pkbden", "bbnpok", "bbnden", "tgk_total", "dentgk_total"]].sum(axis=1)
vv = D.groupby(["tahun", "kategori"]).nilai_rp.sum().unstack().fillna(0)
for c in ["Aturan & ML", "Aturan saja", "ML saja"]:
    T12[f"nilai_{c}_jutaRp"] = (vv.get(c, 0) / 1e6).round(1).values
T12["hibrida_total"] = T12["aturan"] + T12["ml_saja"]; T12["hibrida_pct"] = (100 * T12["hibrida_total"] / T12["n"]).round(2)
save(T12, "T12_hasil_data_riil_per_tahun")
cross = pd.DataFrame({r: [int(((D[r] == 1) & (D.ml_flag == 1)).sum()), int(D[r].sum())] for r in RULE_IDS}, index=["terdeteksi_ML", "n_aturan"]).T
cross["persen_juga_ML"] = (100 * cross.terdeteksi_ML / cross.n_aturan.replace(0, np.nan)).round(1)
save(cross.reset_index().rename(columns={"index": "aturan"}), "S07_irisan_aturan_ML")

top = D.sort_values("hyb_score", ascending=False)
mlonly = D[D.kategori == "ML saja"].sort_values("s_ENS", ascending=False)
def top_table(sub, n):
    idx = sub.index[:n]; pos = D.index.get_indexer(idx)
    reasons = detF.reasons(pos)
    t_ = sub.loc[idx, ["vid", "tahun", "kd_jen_mohon", "kd_jen_mutasi", "kd_jenis_kb", "pkbpok", "pkbden", "bbnpok", "tgk_total", "dentgk_total", "s_ENS"]].copy()
    t_["aturan_dilanggar"] = [",".join(r for r in RULE_IDS if D.at[i, r] == 1) or "-" for i in idx]
    t_["alasan_AE_top3"] = reasons
    return t_
T13 = save(top_table(top, 20).round(4), "T13_top20_anomali_hibrida")
T13b = save(top_table(mlonly, 20).round(4), "T13b_top20_anomali_ML_saja")
# profil temuan ML-saja (tanpa aturan): fitur paling sering jadi alasan
pos = D.index.get_indexer(mlonly.index[:2000])
rs = pd.Series(", ".join(detF.reasons(pos, top=1)).split(", ")).value_counts()
save(rs.rename_axis("fitur_alasan_utama").reset_index(name="n").head(15), "S08_alasan_utama_ML_saja")
D.drop(columns=["tgl"]).to_csv(os.path.join(DP, "pkb_2021_2024_scored_pseudonymised.csv.gz"), index=False, compression="gzip")
log(f"[10] riil: aturan {D.rule_flag.sum():,} | ML {D.ml_flag.sum():,} | ML-saja {(D.kategori=='ML saja').sum():,}")


## [11] Gambar (ID & EN)

In [ ]:
# %% [11] Gambar (ID & EN) ---------------------------------------------------------------------
def flowchart(lang):
    fig, ax = plt.subplots(figsize=(7.4, 10.8)); ax.set_xlim(0, 10); ax.set_ylim(0, 15.2); ax.axis("off"); ax.grid(False)
    def box(x, y, w, h, txt, fc="#EAF2FB", ec="#0072B2", shape="round", fs=7.0, bold=False):
        if shape == "diamond":
            ax.add_patch(plt.Polygon([[x, y + h / 2], [x + w / 2, y + h], [x + w, y + h / 2], [x + w / 2, y]], fc=fc, ec=ec, lw=1.2))
        else:
            ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.15", fc=fc, ec=ec, lw=1.2))
        ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=fs, weight="bold" if bold else "normal", linespacing=1.25)
    def arr(x1, y1, x2, y2, txt=None, dx=.12):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color="#333", lw=1))
        if txt: ax.text((x1 + x2) / 2 + dx, (y1 + y2) / 2, txt, fontsize=6.8, color="#333", weight="bold")
    box(1.3, 14.35, 7.4, .7, T(lang, "MULAI: ekspor ISQL-WEB SAMSAT Mei–Juni 2021–2024\n(4 file, 193.899 transaksi, 24 atribut)",
                              "START: ISQL-WEB SAMSAT export May–June 2021–2024\n(4 files, 193,899 transactions, 24 attributes)"), fc="#FFF3E0", ec="#E69F00", bold=True)
    box(.6, 13.25, 8.8, .75, T(lang, "Pra-pemrosesan: buang baris header/footer, parsing angka & tanggal,\npseudonim SHA-256 nomor polisi, variabel turunan (tunggakan, denda, kelas kendaraan)",
                              "Pre-processing: drop header/footer rows, parse numbers & dates,\nSHA-256 plate pseudonym, derived variables (arrears, penalties, vehicle class)"))
    box(.6, 12.15, 8.8, .75, T(lang, "Deteksi pergeseran skema kode (Jensen–Shannon divergence per atribut per tahun)\n→ harmonisasi kd_status 5/6→3/4 dan kd_jen_mohon 2|x→1|x (2024)",
                              "Coding-scheme drift detection (Jensen–Shannon divergence per field per year)\n→ harmonise kd_status 5/6→3/4 and kd_jen_mohon 2|x→1|x (2024)"))
    arr(5, 14.35, 5, 14.0); arr(5, 13.25, 5, 12.9)
    box(.1, 10.3, 4.85, 1.5, T(lang, "LAPISAN 1 – Mesin aturan domain (R01–R11)\ndenda tanpa tunggakan, kode mutasi tidak\nkonsisten, nilai PKB janggal per jenis, tarif\ndenda, BBNKB vs kode, duplikat\n(profil kode & nilai dari 2021–2023)",
                               "LAYER 1 – Domain rule engine (R01–R11)\npenalty w/o arrears, inconsistent mutation\ncode, implausible PKB per type, penalty\nrate, BBNKB vs code, duplicates\n(code & value profile from 2021–2023)"), fc="#E8F5E9", ec="#009E73")
    box(5.05, 10.3, 4.85, 1.5, T(lang, "Profil kualitas data 6 dimensi:\nkelengkapan, validitas, konsistensi,\nakurasi, plausibilitas, keunikan\n→ Indeks DQ per tahun",
                                "Six-dimension data-quality profile:\ncompleteness, validity, consistency,\naccuracy, plausibility, uniqueness\n→ yearly DQ index"), fc="#E8F5E9", ec="#009E73")
    arr(5, 12.15, 2.5, 11.8); arr(5, 12.15, 7.5, 11.8)
    box(1.0, 8.75, 8.0, .95, T(lang, "Transaksi bersih (tanpa pelanggaran & bukan nihil) → latih 2021–2023, uji 2024\nFitur dasar (nilai, frekuensi kode, status) + fitur domain (rasio denda, z-jenis, z-BBN, BBN×kode)",
                              "Clean records (no rule hit, non-zero) → train 2021–2023, test 2024\nBase features (amounts, code frequency, status) + domain features (penalty ratios, type-z, BBN-z, BBN×code)"), fs=6.7)
    arr(2.5, 10.3, 3.2, 9.7)
    box(.2, 7.15, 3.0, .9, "Isolation Forest\n(300 trees, ψ = 512)", fc="#F3E5F5", ec="#CC79A7")
    box(3.5, 7.15, 3.0, .9, "Local Outlier Factor\n(novelty, k = 35)", fc="#F3E5F5", ec="#CC79A7")
    box(6.8, 7.15, 3.0, .9, "Autoencoder MLP\n(32-8-32, MSE)", fc="#F3E5F5", ec="#CC79A7")
    for x in (1.7, 5, 8.3): arr(5, 8.75, x, 8.05)
    box(1.3, 5.75, 7.4, .85, T(lang, "LAPISAN 2 – Ensemble ML: maksimum persentil skor tiap detektor\n(dikalibrasi terhadap distribusi skor data latih)",
                              "LAYER 2 – ML ensemble: maximum score percentile across detectors\n(calibrated on the training-score distribution)"), fc="#F3E5F5", ec="#CC79A7")
    for x in (1.7, 5, 8.3): arr(x, 7.15, 5, 6.6)
    box(3.4, 4.1, 3.2, 1.2, T(lang, "Melanggar\naturan?", "Rule\nviolated?"), shape="diamond", fc="#FFF3E0", ec="#E69F00", fs=7.5, bold=True)
    arr(5, 5.75, 5, 5.3)
    ax.plot([.25, .25], [10.3, 4.7], color="#333", lw=1); arr(.25, 4.7, 3.4, 4.7)
    ax.text(.35, 7.0, T(lang, "bendera\naturan", "rule\nflags"), fontsize=6.3, color="#333", rotation=90, va="center")
    box(.6, 2.45, 3.6, .9, T(lang, "Prioritas 1:\nskor = 1 + skor ensemble", "Priority 1:\nscore = 1 + ensemble score"), fc="#FDECEA", ec="#D55E00")
    box(5.8, 2.45, 3.6, .9, T(lang, "Skor = skor ensemble\n(bendera bila > P99 data latih)", "Score = ensemble score\n(flag if > training P99)"))
    arr(3.9, 4.35, 2.4, 3.35, T(lang, "ya", "yes"), -.45); arr(6.1, 4.35, 7.6, 3.35, T(lang, "tidak", "no"))
    box(.3, 1.0, 9.4, 1.0, T(lang, "LAPISAN 3 – Skor hibrida → daftar prioritas audit + kode alasan (aturan & galat rekonstruksi AE)\nEvaluasi: injeksi 8 tipe anomali × 10 ulangan; ROC-AUC, PR-AUC, P@k, Recall@5%, F1; uji Friedman–Wilcoxon",
                            "LAYER 3 – Hybrid score → audit priority list + reason codes (rules & AE reconstruction error)\nEvaluation: 8 injected anomaly types × 10 runs; ROC-AUC, PR-AUC, P@k, Recall@5%, F1; Friedman–Wilcoxon"), fc="#FFF3E0", ec="#E69F00", fs=6.7)
    arr(2.4, 2.45, 3.5, 2.0); arr(7.6, 2.45, 6.5, 2.0)
    box(2.3, .05, 5.4, .6, T(lang, "SELESAI: dasbor audit & umpan balik petugas", "END: audit dashboard & officer feedback"), fc="#FFF3E0", ec="#E69F00", bold=True)
    arr(5, 1.0, 5, .65)
    savefig(fig, "F01_flowchart_metode", lang)

def architecture(lang):
    fig, ax = plt.subplots(figsize=(7.4, 3.0)); ax.set_xlim(0, 15); ax.set_ylim(0, 5.4); ax.axis("off"); ax.grid(False)
    items = [(0.2, T(lang, "Sumber data\nISQL-WEB SAMSAT\n(transaksi harian)", "Data source\nISQL-WEB SAMSAT\n(daily transactions)"), "#FFF3E0", "#E69F00"),
             (3.2, T(lang, "ETL &\npseudonimisasi\n(batch harian)", "ETL &\npseudonymisation\n(daily batch)"), "#EAF2FB", "#0072B2"),
             (6.2, T(lang, "Mesin aturan\nR01–R11\n+ profil kode", "Rule engine\nR01–R11\n+ code profile"), "#E8F5E9", "#009E73"),
             (9.2, T(lang, "Skoring ML\nIF · LOF · AE\n→ ensemble", "ML scoring\nIF · LOF · AE\n→ ensemble"), "#F3E5F5", "#CC79A7"),
             (12.2, T(lang, "Dasbor audit\nprioritas + alasan\n+ indeks DQ", "Audit dashboard\npriority + reasons\n+ DQ index"), "#FDECEA", "#D55E00")]
    for x, t_, fc, ec in items:
        ax.add_patch(FancyBboxPatch((x, 2.3), 2.6, 2.2, boxstyle="round,pad=0.03,rounding_size=0.2", fc=fc, ec=ec, lw=1.3))
        ax.text(x + 1.3, 3.4, t_, ha="center", va="center", fontsize=7.0)
    for x in [2.8, 5.8, 8.8, 11.8]:
        ax.annotate("", xy=(x + .4, 3.4), xytext=(x, 3.4), arrowprops=dict(arrowstyle="-|>", lw=1.3))
    ax.plot([13.5, 13.5, 7.5], [2.3, 1.4, 1.4], color="#D55E00", lw=1.1)
    ax.annotate("", xy=(7.5, 2.3), xytext=(7.5, 1.4), arrowprops=dict(arrowstyle="-|>", lw=1.1, color="#D55E00"))
    ax.plot([10.5, 10.5], [1.4, 2.3], color="#D55E00", lw=1.1, ls=":")
    ax.text(10.5, .75, T(lang, "Umpan balik petugas (label konfirmasi) → pemutakhiran aturan & pelatihan ulang berkala",
                         "Officer feedback (confirmed labels) → rule update & periodic retraining"), ha="center", fontsize=7.6, color="#D55E00")
    savefig(fig, "F02_arsitektur_sistem", lang)

for lg in ["id", "en"]:
    flowchart(lg); architecture(lg)
    # F03 prevalensi aturan per tahun (heatmap %)
    H = T03.set_index("tahun")[[r + "_pct" for r in RULE_IDS]]; H.columns = [f"{r} {RNAME[lg][r]}" for r in RULE_IDS]
    fig, ax = plt.subplots(figsize=(7.2, 4.2)); im = ax.imshow(np.log10(H.T.values + 1e-3), cmap="Oranges", aspect="auto")
    for i in range(H.shape[1]):
        for j in range(H.shape[0]):
            ax.text(j, i, f"{H.iloc[j, i]:.3f}%\n(n={int(T03.iloc[j][RULE_IDS[i]]):,})", ha="center", va="center", fontsize=6.3)
    ax.set_xticks(range(4)); ax.set_xticklabels(YEARS); ax.set_yticks(range(len(RULE_IDS))); ax.set_yticklabels(H.columns, fontsize=7.5); ax.grid(False)
    ax.set_title(T(lg, "Prevalensi pelanggaran aturan domain per tahun", "Domain-rule violation prevalence by year"))
    savefig(fig, "F03_prevalensi_aturan", lg)
    # F04 skor DQ
    dims_en = {"Kelengkapan": "Completeness", "Validitas": "Validity", "Konsistensi": "Consistency", "Akurasi": "Accuracy",
               "Plausibilitas": "Plausibility", "Keunikan": "Uniqueness", "Indeks DQ": "DQ index"}
    Q = T04.copy(); Q.columns = [c if lg == "id" else dims_en[c] for c in Q.columns]
    fig, ax = plt.subplots(figsize=(7.2, 2.6)); im = ax.imshow(Q.values, cmap="RdYlGn", vmin=97, vmax=100, aspect="auto")
    for i in range(Q.shape[0]):
        for j in range(Q.shape[1]): ax.text(j, i, f"{Q.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7.5, color="white" if Q.iloc[i, j] > 99.6 or Q.iloc[i, j] < 97.3 else "black")
    ax.set_xticks(range(Q.shape[1])); ax.set_xticklabels(Q.columns, fontsize=7.5, rotation=15); ax.set_yticks(range(4)); ax.set_yticklabels(YEARS); ax.grid(False)
    plt.colorbar(im, ax=ax, fraction=.025, label=T(lg, "skor (%)", "score (%)"))
    ax.set_title(T(lg, "Kartu skor kualitas data SAMSAT per dimensi", "SAMSAT data-quality scorecard by dimension"))
    savefig(fig, "F04_skor_kualitas_data", lg)
    # F05 distribusi pokok per jenis + batas R09
    top_codes = D.kd_jenis_kb.value_counts().index[:8]
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    data = [np.log10(D.loc[(D.kd_jenis_kb == c) & (D.pkbpok > 0), "pkbpok"]) for c in top_codes]
    ax.boxplot(data, tick_labels=top_codes, showfliers=False, widths=.5, patch_artist=True, boxprops=dict(facecolor="#EAF2FB", edgecolor="#0072B2"))
    for i, c in enumerate(top_codes):
        sub = D[(D.kd_jenis_kb == c) & (D.pkbpok > 0)]
        out_ = sub[sub.R09 == 1]; ins = sub[sub.R09 == 0].sample(min(400, (sub.R09 == 0).sum()), random_state=1)
        ax.scatter(np.random.normal(i + 1, .06, len(ins)), np.log10(ins.pkbpok), s=2, color="#999", alpha=.4)
        ax.scatter(np.random.normal(i + 1, .06, len(out_)), np.log10(out_.pkbpok), s=7, color="#D55E00", label="R09" if i == 0 else None)
        g = grp_stats.loc[c]; ax.hlines([g.med - 3.5 * g.mad, g.med + 3.5 * g.mad], i + .7, i + 1.3, colors="#D55E00", linestyles="--", lw=.8)
    ax.set_ylabel("log10 PKB pokok (Rp)"); ax.set_xlabel(T(lg, "kode jenis kendaraan (kd_jenis_kb)", "vehicle-type code (kd_jenis_kb)"))
    ax.legend(loc="upper right"); ax.set_title(T(lg, "Nilai PKB per jenis kendaraan dan batas robust-z ±3,5 (R09)", "PKB value by vehicle type and robust-z ±3.5 limits (R09)"))
    savefig(fig, "F05_nilai_pkb_per_jenis", lg)
    # F06 matriks kode mohon x mutasi
    ct = pd.crosstab(D.kd_jen_mohon, D.kd_jen_mutasi)
    fig, ax = plt.subplots(figsize=(6.4, 3.4)); ax.imshow(np.log10(ct.values + 1), cmap="Blues", aspect="auto"); ax.grid(False)
    for i, a_ in enumerate(ct.index):
        for j, b_ in enumerate(ct.columns):
            v = ct.iloc[i, j]
            if v:
                bad = f"{a_}|{b_}" in RARE_PAIR or code_cnt["kd_jen_mohon"].get(a_, 0) < 50 or code_cnt["kd_jen_mutasi"].get(b_, 0) < 50
                ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=6.5, color="#D55E00" if bad else "black", weight="bold" if bad else "normal")
                if bad: ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec="#D55E00", lw=1.2))
    ax.set_xticks(range(len(ct.columns))); ax.set_xticklabels(ct.columns); ax.set_yticks(range(len(ct.index))); ax.set_yticklabels(ct.index)
    ax.set_xlabel("kd_jen_mutasi"); ax.set_ylabel("kd_jen_mohon")
    ax.set_title(T(lg, "Matriks kode permohonan × mutasi (kotak merah = tidak konsisten, R07)", "Application × mutation code matrix (red = inconsistent, R07)"))
    savefig(fig, "F06_matriks_kode_mutasi", lg)
    # F07 ROC & PR
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.3))
    for i, m in enumerate(METHODS):
        y_, s_ = curves[m]; f_, t_, _ = roc_curve(y_, s_); p_, r_, _ = precision_recall_curve(y_, s_)
        axs[0].plot(f_, t_, color=PAL[i], lw=1.4 if m == "Hibrida" else 1, label=f"{MLAB[lg][m]} ({roc_auc_score(y_, s_):.3f})")
        axs[1].plot(r_, p_, color=PAL[i], lw=1.4 if m == "Hibrida" else 1, label=f"{MLAB[lg][m]} ({average_precision_score(y_, s_):.3f})")
    axs[0].plot([0, 1], [0, 1], "k:", lw=.6); axs[0].set_xlabel("FPR"); axs[0].set_ylabel("TPR"); axs[0].set_title("ROC (AUC)")
    axs[1].set_xlabel("Recall"); axs[1].set_ylabel(T(lg, "Presisi", "Precision")); axs[1].set_title("Precision–Recall (AP)")
    for a_ in axs: a_.legend(fontsize=6.5, loc="lower right" if a_ is axs[0] else "lower left")
    savefig(fig, "F07_kurva_ROC_PR", lg)
    # F08 heatmap recall per tipe
    M8 = T08.copy(); M8.index = [f"{i[0]} {i[1] if lg == 'id' else i[2]}" for i in INJ]; M8.columns = [MLAB[lg][m] for m in METHODS]
    fig, ax = plt.subplots(figsize=(7.2, 3.6)); im = ax.imshow(M8.values, cmap="viridis", vmin=0, vmax=1, aspect="auto"); ax.grid(False)
    for i in range(M8.shape[0]):
        for j in range(M8.shape[1]): ax.text(j, i, f"{M8.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if M8.iloc[i, j] < .6 else "black")
    ax.set_xticks(range(M8.shape[1])); ax.set_xticklabels(M8.columns); ax.set_yticks(range(M8.shape[0])); ax.set_yticklabels(M8.index, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=.03, label="Recall@k")
    ax.set_title(T(lg, "Recall@k per tipe anomali (rerata 10 ulangan)", "Recall@k by anomaly type (mean of 10 runs)"))
    savefig(fig, "F08_recall_per_tipe", lg)
    # F09 bar komparasi metrik
    fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.8))
    for a_, m_ in zip(axs, ["PR_AUC", "P@k", "F1"]):
        mu = agg[(m_, "mean")].values; sdv = agg[(m_, "std")].values
        a_.bar(range(len(METHODS)), mu, yerr=sdv, color=[PAL[i] for i in range(len(METHODS))], capsize=2)
        a_.set_xticks(range(len(METHODS))); a_.set_xticklabels([MLAB[lg][m] for m in METHODS], rotation=45, ha="right", fontsize=7)
        a_.set_title(m_.replace("_", "-")); a_.set_ylim(0, 1.05)
        for i, v in enumerate(mu): a_.text(i, v + .03, f"{v:.2f}", ha="center", fontsize=6.3)
    fig.suptitle(T(lg, "Komparasi kinerja detektor (rerata ± sd, 10 ulangan)", "Detector performance comparison (mean ± sd, 10 runs)"), fontsize=9.5, weight="bold", y=1.03)
    savefig(fig, "F09_komparasi_metrik", lg)
    # F10 distribusi skor detektor (AE & IF) normal vs tipe anomali
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.9))
    groups = ["normal"] + [i[0] for i in INJ]
    for a_, k_, lab_ in [(axs[0], "AE", T(lg, "log10 galat rekonstruksi AE", "log10 AE reconstruction error")), (axs[1], "IF", T(lg, "skor anomali IF", "IF anomaly score"))]:
        v_ = [np.log10(S0[k_].values[te0.typ.values == g] + 1e-6) if k_ == "AE" else S0[k_].values[te0.typ.values == g] for g in groups]
        parts = a_.violinplot(v_, showmedians=True, widths=.8)
        for j_, pc in enumerate(parts["bodies"]): pc.set_facecolor("#BBBBBB" if j_ == 0 else PAL[1]); pc.set_alpha(.6)
        thr = np.quantile(det0.ref[k_], .99); a_.axhline(np.log10(thr) if k_ == "AE" else thr, color="#D55E00", ls="--", lw=.9, label=T(lg, "ambang P99 latih", "training P99"))
        a_.set_xticks(range(1, len(groups) + 1)); a_.set_xticklabels(groups, fontsize=7); a_.set_ylabel(lab_); a_.legend(fontsize=6.5, loc="upper left")
    fig.suptitle(T(lg, "Distribusi skor: transaksi normal vs 8 tipe anomali injeksi (seed 0)", "Score distribution: normal vs eight injected anomaly types (seed 0)"), fontsize=9.5, weight="bold", y=1.03)
    savefig(fig, "F10_distribusi_skor", lg)
    # F11 sensitivitas
    fig, axs = plt.subplots(1, 5, figsize=(8.6, 2.4))
    for a_, (p_, g_) in zip(axs, T10.groupby("parameter", sort=False)):
        a_.plot(range(len(g_)), g_["PR_AUC"], "o-", label=g_["metode"].iloc[0], color="#0072B2")
        a_.plot(range(len(g_)), g_["PR_AUC_hibrida"], "s--", label=MLAB[lg]["Hibrida"], color="#D55E00")
        a_.set_xticks(range(len(g_))); a_.set_xticklabels([str(v).replace("max persentil (dipakai)", "max").replace("rerata persentil", T(lg, "rerata", "mean")) for v in g_["nilai"]], rotation=45, fontsize=6.5); a_.set_ylim(0, 1.02)
        a_.set_title({"LOF n_neighbors": "LOF k", "IF n_estimators": "IF trees", "AE arsitektur": T(lg, "AE arsitektur", "AE architecture"),
                      "Ukuran data latih": T(lg, "n latih", "n train"), "Strategi fusi": T(lg, "fusi", "fusion")}[p_], fontsize=8); a_.legend(fontsize=5.8, loc="lower right")
    axs[0].set_ylabel("PR-AUC"); fig.suptitle(T(lg, "Analisis sensitivitas hiperparameter", "Hyper-parameter sensitivity"), fontsize=9.5, weight="bold", y=1.05)
    savefig(fig, "F11_sensitivitas", lg)
    # F12 ablasi fitur
    fig, ax = plt.subplots(figsize=(6.0, 2.8)); ms = ["IF", "LOF", "AE", "Ensemble ML", "Hibrida"]; w = .38
    for j, (fs_, c_) in enumerate([("dasar", "#56B4E9"), ("dasar+domain", "#0072B2")]):
        mu = [T11.loc[m, ("mean", fs_)] for m in ms]; sdv = [T11.loc[m, ("std", fs_)] for m in ms]
        ax.bar(np.arange(len(ms)) + (j - .5) * w, mu, w, yerr=sdv, capsize=2, color=c_,
               label={"dasar": T(lg, "fitur dasar", "base features"), "dasar+domain": T(lg, "dasar + domain", "base + domain")}[fs_])
        for i, v in enumerate(mu): ax.text(i + (j - .5) * w, v + .02, f"{v:.2f}", ha="center", fontsize=6.3)
    ax.set_xticks(range(len(ms))); ax.set_xticklabels([MLAB[lg][m] for m in ms]); ax.set_ylabel("PR-AUC"); ax.set_ylim(0, 1.08); ax.legend(loc="upper left")
    ax.set_title(T(lg, "Ablasi: kontribusi fitur berbasis domain", "Ablation: contribution of domain features"))
    savefig(fig, "F12_ablasi_fitur", lg)
    # F13 temuan pada data riil: komposisi per tahun + irisan aturan-ML
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.0))
    cats = ["Aturan saja", "Aturan & ML", "ML saja", "Nihil (informatif)"]; cen = {"Aturan saja": "Rules only", "Aturan & ML": "Rules & ML", "ML saja": "ML only", "Nihil (informatif)": "Zero-value (informative)"}
    bottom = np.zeros(4)
    for i, c in enumerate(cats):
        v = D[D.kategori == c].groupby("tahun").size().reindex(YEARS).fillna(0).values / D.groupby("tahun").size().values * 100
        axs[0].bar(YEARS, v, bottom=bottom, color=[PAL[2], PAL[1], PAL[3], "#BBBBBB"][i], label=c if lg == "id" else cen[c]); bottom += v
    axs[0].set_ylabel(T(lg, "% transaksi", "% of transactions")); axs[0].legend(fontsize=6.5); axs[0].set_xticks(YEARS)
    axs[0].set_title(T(lg, "Transaksi terflag (data riil)", "Flagged transactions (real data)"))
    cr = cross[cross.n_aturan > 0]
    axs[1].barh(range(len(cr)), cr["persen_juga_ML"], color="#CC79A7"); axs[1].set_yticks(range(len(cr))); axs[1].set_yticklabels(cr.index, fontsize=7)
    for i, (v, n_) in enumerate(zip(cr["persen_juga_ML"], cr["n_aturan"])): axs[1].text(v + 1, i, f"{v:.0f}% (n={n_:,})", va="center", fontsize=6.3)
    axs[1].set_xlim(0, 125); axs[1].set_xlabel(T(lg, "% juga ditandai ML", "% also flagged by ML"))
    axs[1].set_title(T(lg, "Irisan aturan dan ML", "Rule–ML overlap"))
    savefig(fig, "F13_temuan_data_riil", lg)
    # F14 proyeksi PCA
    smp = D.sample(30000, random_state=3); Zs = np.clip(detF.sc.transform(make_features(smp)[detF.cols]), -10, 10)
    P = PCA(2, random_state=0).fit(np.clip(detF.sc.transform(make_features(TRAIN.sample(20000, random_state=0))[detF.cols]), -10, 10)).transform(Zs)
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    for c, col, s_, lab in [("Normal", "#BBBBBB", 2, T(lg, "normal", "normal")), ("Aturan saja", PAL[2], 6, T(lg, "aturan saja", "rules only")),
                            ("ML saja", PAL[3], 6, T(lg, "ML saja", "ML only")), ("Aturan & ML", PAL[1], 8, T(lg, "aturan & ML", "rules & ML"))]:
        m_ = smp.kategori.values == c; ax.scatter(P[m_, 0], P[m_, 1], s=s_, color=col, alpha=.5 if c == "Normal" else .85, label=f"{lab} ({m_.sum():,})")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(fontsize=7, markerscale=2)
    ax.set_title(T(lg, "Proyeksi PCA ruang fitur (sampel 30.000 transaksi)", "PCA projection of feature space (30,000-transaction sample)"))
    savefig(fig, "F14_proyeksi_PCA", lg)
    # F15 waktu komputasi
    tt = TM.mean(numeric_only=True)
    fig, ax = plt.subplots(figsize=(5.6, 2.4))
    lab = ["IF", "LOF", "AE"]; ax.bar(np.arange(3) - .2, [tt[f"fit_{k}"] for k in lab], .4, label=T(lg, "latih (40.000)", "fit (40,000)"), color="#0072B2")
    ax.bar(np.arange(3) + .2, [tt[f"score_{k}"] for k in lab], .4, label=T(lg, f"skoring ({N_NORM + 8 * N_PER:,})", f"scoring ({N_NORM + 8 * N_PER:,})"), color="#E69F00")
    for i_, k_ in enumerate(lab):
        ax.text(i_ - .2, tt[f"fit_{k_}"] + .08, f"{tt[f'fit_{k_}']:.2f}", ha="center", fontsize=6.5); ax.text(i_ + .2, tt[f"score_{k_}"] + .08, f"{tt[f'score_{k_}']:.2f}", ha="center", fontsize=6.5)
    ax.set_xticks(range(3)); ax.set_xticklabels(lab); ax.set_ylabel(T(lg, "detik", "seconds")); ax.legend()
    ax.set_title(T(lg, "Waktu komputasi rata-rata (CPU 2 inti)", "Mean computation time (2-core CPU)"))
    savefig(fig, "F15_waktu_komputasi", lg)
    # F16 pergeseran kode
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.9))
    d24 = DRIFT[DRIFT.tahun == 2024].set_index("field")
    xx = np.arange(len(d24))
    axs[0].bar(xx - .2, d24["JSD_mentah"] + 1e-4, .4, color="#D55E00", label=T(lg, "kode mentah", "raw codes"))
    axs[0].bar(xx + .2, d24["JSD_harmonisasi"] + 1e-4, .4, color="#009E73", label=T(lg, "setelah harmonisasi", "after harmonisation"))
    axs[0].set_yscale("log"); axs[0].set_xticks(xx); axs[0].set_xticklabels([f.replace("kd_jen_mohon|kd_jen_mutasi", "mohon|mutasi") for f in d24.index], rotation=25, fontsize=7)
    axs[0].set_ylabel("JSD (bit)"); axs[0].legend(fontsize=6.5); axs[0].set_title(T(lg, "Divergensi kode 2024 vs 2021–2023", "Code divergence 2024 vs 2021–2023"))
    for pr_, c_ in [("1|7", PAL[0]), ("2|7", PAL[1]), ("1|9", PAL[2]), ("2|9", PAL[3])]:
        axs[1].plot(YEARS, [((D.tahun == y) & (D.pair_raw == pr_)).sum() for y in YEARS], "o-", color=c_, label=f"mohon|mutasi {pr_}")
    axs[1].plot(YEARS, [((D.tahun == y) & D.kd_status_raw.isin(["5", "6"])).sum() / 5 for y in YEARS], "s--", color="#555", label=T(lg, "kd_status 5/6 (÷5)", "kd_status 5/6 (÷5)"))
    axs[1].set_xticks(YEARS); axs[1].set_ylabel(T(lg, "jumlah transaksi", "transactions")); axs[1].legend(fontsize=6)
    axs[1].set_title(T(lg, "Perubahan skema kode pada 2024", "Coding-scheme change in 2024"))
    savefig(fig, "F16_pergeseran_kode", lg)
log("[11] gambar selesai")


## [12] Ringkasan

In [ ]:
# %% [12] Ringkasan --------------------------------------------------------------------------
summary = {"n_transaksi": int(len(D)), "n_train_bersih": int(len(TRAIN)), "n_uji_bersih": int(len(TESTB)),
           "rule_flag": int(D.rule_flag.sum()), "rule_flag_pct": float(D.rule_flag.mean() * 100),
           "ml_flag": int(D.ml_flag.sum()), "ml_only": int((D.kategori == "ML saja").sum()),
           "zero_value": int(D.zero_value.sum()), "plate_nonstandard": int(D.v_plate_fmt.sum()),
           "dq_index": T04["Indeks DQ"].round(3).to_dict(), "friedman_p": float(fr_p),
           "pr_auc": agg[("PR_AUC", "mean")].round(4).to_dict(), "roc_auc": agg[("ROC_AUC", "mean")].round(4).to_dict(),
           "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
           "n_seeds": N_SEEDS, "runtime_s": round(time.time() - t0, 1)}
json.dump(summary, open(os.path.join(OUT, "results", "run_summary.json"), "w"), indent=2, default=str)
open(os.path.join(OUT, "results", "run_log.txt"), "w").write("\n".join(LOG))
log(f"Selesai dalam {time.time()-t0:.0f}s")


## [13] Tampilkan tabel utama & gambar

In [ ]:
from IPython.display import Image, display
for t in ['T07_perbandingan_metode','T08_recall_per_tipe_anomali','T09_uji_statistik','T12_hasil_data_riil_per_tahun']:
    print(t); display(pd.read_csv(os.path.join(TAB, t + '.csv')))
for f in ['F01_flowchart_metode','F09_komparasi_metrik','F08_recall_per_tipe','F13_temuan_data_riil','F16_pergeseran_kode']:
    display(Image(os.path.join(FIGD['id'], f + '.png'), width=700))

## [14] (Opsional) Susun naskah DOCX dari hasil Colab
Membutuhkan `Template Inovatif 2022 ok.docx` (versi .docx) di folder `project4`, `src`, `data`, atau folder induk.

In [ ]:
cands = [os.path.join(d, 'Template Inovatif 2022 ok.docx') for d in (OUT_DIR, SRC_DIR, DATA_DIR, BASE_DIR)]
tpl = next((c for c in cands if os.path.exists(c)), None)
if tpl is None:
    print('Template .docx tidak ditemukan; lewati penyusunan naskah. Lokasi yang dicek:', cands)
else:
    !python "$SRC_DIR/build_manuscript.py" --root "$OUT_DIR" --template "$tpl" --out_dir "$OUT_DIR/manuscript" --lang both
    print(sorted(os.listdir(os.path.join(OUT_DIR, 'manuscript'))))